# Face_Recognisation

## face rec_pred

In [1]:
# Load the final clean database

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

EMBEDDINGS_DIR = Path("./Data/Embeddings")

EMBEDDING_FILE = EMBEDDINGS_DIR / "embeddings_clean.npy"
METADATA_FILE = EMBEDDINGS_DIR / "metadata_clean.csv"

recognition_embeddings = np.load(EMBEDDING_FILE)
recognition_metadata = pd.read_csv(METADATA_FILE)

print("Embeddings shape :", recognition_embeddings.shape)
print("Metadata shape   :", recognition_metadata.shape)
print("Number of people :", recognition_metadata["person"].nunique())

Embeddings shape : (211, 512)
Metadata shape   : (211, 3)
Number of people : 14


In [3]:
# Set the final threshold
FINAL_THRESHOLD = 0.36

print(f"Final threshold: {FINAL_THRESHOLD:.2f}")

Final threshold: 0.36


**Create the cosine similarity function** :

Because your ArcFace embeddings are normalized, their dot product is equivalent to cosine similarity.

In [8]:
def cosine_similarity(embedding1, embedding2):
    return float(
        np.dot(embedding1, embedding2)
    )

**Create the core recognition function**

In [9]:
def recognize_embedding(
    query_embedding,
    database_embeddings,
    database_metadata,
    threshold=FINAL_THRESHOLD
):
    
    # Calculate similarity with every stored embedding
    similarities = np.dot(
        database_embeddings,
        query_embedding
    )

    # Find highest similarity
    best_index = np.argmax(similarities)
    best_similarity = float(similarities[best_index])

    # Get corresponding person
    best_person = database_metadata.iloc[
        best_index
    ]["person"]

    # Apply threshold
    if best_similarity >= threshold:
        status = "KNOWN"
        identity = best_person
    else:
        status = "UNKNOWN"
        identity = "Unknown"

    return {
        "identity": identity,
        "similarity": best_similarity,
        "status": status,
        "best_index": int(best_index)
    }

**Test the recognition engine with an existing image**

Before touching the camera, we should test the core recognition algorithm using one of your existing embeddings.

In [10]:
test_embedding = recognition_embeddings[0]

result = recognize_embedding(
    test_embedding,
    recognition_embeddings,
    recognition_metadata
)

print("Identity  :", result["identity"])
print("Similarity:", f'{result["similarity"]:.4f}')
print("Status    :", result["status"])

Identity  : Abhishek_Halagi
Similarity: 1.0000
Status    : KNOWN


**More meaningful test: leave-one-out recognition**

The previous test is not a realistic test, because the query image itself exists in the database.

In [11]:
def recognize_leave_one_out(
    index,
    database_embeddings,
    database_metadata,
    threshold=FINAL_THRESHOLD
):
    
    query_embedding = database_embeddings[index]

    mask = np.ones(
        len(database_embeddings),
        dtype=bool
    )

    mask[index] = False

    reference_embeddings = database_embeddings[mask]
    reference_metadata = database_metadata.iloc[
        np.where(mask)[0]
    ].reset_index(drop=True)

    return recognize_embedding(
        query_embedding,
        reference_embeddings,
        reference_metadata,
        threshold
    )

In [12]:
test_index = 0

result = recognize_leave_one_out(
    test_index,
    recognition_embeddings,
    recognition_metadata
)

actual_person = recognition_metadata.iloc[
    test_index
]["person"]

print("Actual identity :", actual_person)
print("Predicted        :", result["identity"])
print("Similarity       :", f'{result["similarity"]:.4f}')
print("Status           :", result["status"])

Actual identity : Abhishek_Halagi
Predicted        : Abhishek_Halagi
Similarity       : 0.8082
Status           : KNOWN


## CELLS

In [13]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

print("OpenCV version :", cv2.__version__)
print("NumPy version  :", np.__version__)
print("Pandas version :", pd.__version__)

OpenCV version : 5.0.0
NumPy version  : 2.5.2
Pandas version : 3.0.5


In [16]:
# Define project paths
ALIGNED_DIR = Path("./Data/Classmates_Aligned")

EMBEDDINGS_DIR = Path("./Data/Embeddings")

MODEL_PATH = Path(
    "./models/face_detection_yunet_2026may.onnx"
)

EMBEDDING_FILE = (
    EMBEDDINGS_DIR / "embeddings_clean.npy"
)

METADATA_FILE = (
    EMBEDDINGS_DIR / "metadata_clean.csv"
)

print("Aligned directory :", ALIGNED_DIR.resolve())
print("YuNet model       :", MODEL_PATH.resolve())
print("Embedding file    :", EMBEDDING_FILE.resolve())
print("Metadata file     :", METADATA_FILE.resolve())

Aligned directory : D:\Artificial intelligence\projects\FaceRecognisationSystem\001_CLASSMATES_FRM\Data\Classmates_Aligned
YuNet model       : D:\Artificial intelligence\projects\FaceRecognisationSystem\001_CLASSMATES_FRM\models\face_detection_yunet_2026may.onnx
Embedding file    : D:\Artificial intelligence\projects\FaceRecognisationSystem\001_CLASSMATES_FRM\Data\Embeddings\embeddings_clean.npy
Metadata file     : D:\Artificial intelligence\projects\FaceRecognisationSystem\001_CLASSMATES_FRM\Data\Embeddings\metadata_clean.csv


In [17]:
# Verify the important files
print("YuNet model exists :", MODEL_PATH.exists())
print("Embeddings exist   :", EMBEDDING_FILE.exists())
print("Metadata exists    :", METADATA_FILE.exists())

YuNet model exists : True
Embeddings exist   : True
Metadata exists    : True


In [18]:
# Load the clean embedding database
recognition_embeddings = np.load(
    EMBEDDING_FILE
)

recognition_metadata = pd.read_csv(
    METADATA_FILE
)

print("Embeddings shape :", recognition_embeddings.shape)
print("Metadata shape   :", recognition_metadata.shape)
print(
    "Number of people:",
    recognition_metadata["person"].nunique()
)

Embeddings shape : (211, 512)
Metadata shape   : (211, 3)
Number of people: 14


In [19]:
# Verify embedding normalization
embedding_norms = np.linalg.norm(
    recognition_embeddings,
    axis=1
)

print("Minimum norm:", embedding_norms.min())
print("Maximum norm:", embedding_norms.max())
print("Mean norm   :", embedding_norms.mean())

print(
    "Normalized:",
    np.allclose(
        embedding_norms,
        1.0,
        atol=1e-5
    )
)

Minimum norm: 0.9999999
Maximum norm: 1.0000001
Mean norm   : 1.0
Normalized: True


In [20]:
# Set the operating threshold
# We'll use the threshold selected from your clean-dataset evaluation:
FINAL_THRESHOLD = 0.36

print(
    f"Final recognition threshold: "
    f"{FINAL_THRESHOLD:.2f}"
)

Final recognition threshold: 0.36


**Cosine similarity**

Since your ArcFace embeddings are normalized, cosine similarity can be calculated using the dot product.

In [21]:
def cosine_similarity(embedding1, embedding2):
    return float(np.dot(embedding1, embedding2))

In [24]:
test_similarity = cosine_similarity(
    recognition_embeddings[0],
    recognition_embeddings[0]
)

print("Self-similarity:", test_similarity)

Self-similarity: 1.0


**Core recognition function**

Now create the actual matcher:

In [25]:
def recognize_embedding(
    query_embedding,
    database_embeddings,
    database_metadata,
    threshold=FINAL_THRESHOLD
):
    
    # Ensure query is a NumPy float32 vector
    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32
    )

    # Compare query against every stored embedding
    similarities = np.dot(
        database_embeddings,
        query_embedding
    )

    # Find the highest similarity
    best_index = int(np.argmax(similarities))

    best_similarity = float(
        similarities[best_index]
    )

    # Get identity corresponding to best match
    best_person = database_metadata.iloc[
        best_index
    ]["person"]

    # Apply recognition threshold
    if best_similarity >= threshold:

        identity = best_person
        status = "KNOWN"

    else:

        identity = "Unknown"
        status = "UNKNOWN"

    return {
        "identity": identity,
        "similarity": best_similarity,
        "status": status,
        "best_index": best_index
    }

**Basic recognition test**

We'll first deliberately test an embedding that already exists in the database.

In [26]:
test_index = 0

test_embedding = recognition_embeddings[
    test_index
]

result = recognize_embedding(
    query_embedding=test_embedding,
    database_embeddings=recognition_embeddings,
    database_metadata=recognition_metadata
)

print("Actual identity :", recognition_metadata.iloc[test_index]["person"])
print("Predicted       :", result["identity"])
print("Similarity      :", f'{result["similarity"]:.4f}')
print("Status          :", result["status"])

Actual identity : Abhishek_Halagi
Predicted       : Abhishek_Halagi
Similarity      : 1.0000
Status          : KNOWN


### Proper leave-one-out test

In [27]:
def recognize_leave_one_out(
    index,
    database_embeddings,
    database_metadata,
    threshold=FINAL_THRESHOLD
):
    
    query_embedding = database_embeddings[index]

    # Exclude the query itself
    mask = np.ones(
        len(database_embeddings),
        dtype=bool
    )

    mask[index] = False

    reference_embeddings = (
        database_embeddings[mask]
    )

    reference_metadata = (
        database_metadata.iloc[
            np.where(mask)[0]
        ]
        .reset_index(drop=True)
    )

    return recognize_embedding(
        query_embedding=query_embedding,
        database_embeddings=reference_embeddings,
        database_metadata=reference_metadata,
        threshold=threshold
    )

In [28]:
# Test one image properly
test_index = 0

actual_person = recognition_metadata.iloc[
    test_index
]["person"]

result = recognize_leave_one_out(
    index=test_index,
    database_embeddings=recognition_embeddings,
    database_metadata=recognition_metadata
)

print("Actual identity :", actual_person)
print("Predicted       :", result["identity"])
print("Similarity      :", f'{result["similarity"]:.4f}')
print("Status          :", result["status"])

Actual identity : Abhishek_Halagi
Predicted       : Abhishek_Halagi
Similarity      : 0.8082
Status          : KNOWN


### Full 211-image recognition test

In [29]:
results = []

for i in range(len(recognition_embeddings)):

    actual_person = recognition_metadata.iloc[i]["person"]

    result = recognize_leave_one_out(
        index=i,
        database_embeddings=recognition_embeddings,
        database_metadata=recognition_metadata
    )

    results.append({
        "index": i,
        "actual": actual_person,
        "predicted": result["identity"],
        "similarity": result["similarity"],
        "status": result["status"]
    })

recognition_results = pd.DataFrame(results)

print("Total tested:", len(recognition_results))

Total tested: 211


In [30]:
recognition_results["correct"] = (
    recognition_results["actual"] ==
    recognition_results["predicted"]
)

print(
    "Correct:",
    recognition_results["correct"].sum()
)

print(
    "Accuracy:",
    recognition_results["correct"].mean() * 100,
    "%"
)

Correct: 211
Accuracy: 100.0 %


In [32]:
recognition_results[
    ~recognition_results["correct"]
].sort_values(
    "similarity"
)

,index,actual,predicted,similarity,status,correct


In [33]:
recognition_results[
    recognition_results["correct"]
].sort_values(
    "similarity"
).head(20)

,index,actual,predicted,similarity,status,correct
88,88,Mahesh_Oulkar,Mahesh_Oulkar,0.401812,KNOWN,True
181,181,Sahil_Patil,Sahil_Patil,0.469103,KNOWN,True
134,134,Rohit_Patil,Rohit_Patil,0.487796,KNOWN,True
152,152,Sahil_Kurbet,Sahil_Kurbet,0.519254,KNOWN,True
208,208,Vishwanath_Muchandi,Vishwanath_Muchandi,0.521854,KNOWN,True
150,150,Rohit_Patil,Rohit_Patil,0.533462,KNOWN,True
119,119,Pranav_Kavale,Pranav_Kavale,0.556386,KNOWN,True
156,156,Sahil_Kurbet,Sahil_Kurbet,0.576242,KNOWN,True
207,207,Vishwanath_Muchandi,Vishwanath_Muchandi,0.578317,KNOWN,True
47,47,Dhurvankur_,Dhurvankur_,0.580737,KNOWN,True


## Building Pipeline

In [37]:
import insightface

In [38]:
# Load YuNet
detector = cv2.FaceDetectorYN.create(
    model=str(MODEL_PATH),
    config="",
    input_size=(320, 320),
    score_threshold=0.6,
    nms_threshold=0.3,
    top_k=5000
)

print("YuNet loaded.")

YuNet loaded.


In [39]:
# Load ArcFace
from insightface.model_zoo import get_model

recognition_model = insightface.model_zoo.get_model(
    "buffalo_l",
    providers=["CPUExecutionProvider"]
)

recognition_model.prepare(ctx_id=0)

print("ArcFace loaded.")

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
ArcFace loaded.


**YuNet detection + alignment**

Use your existing alignment function that you already verified visually.

In [40]:
def detect_and_align(image):

    h, w = image.shape[:2]

    detector.setInputSize((w, h))

    _, faces = detector.detect(image)

    if faces is None or len(faces) == 0:
        return None

    # Select highest-confidence face
    face = max(
        faces,
        key=lambda x: x[14]
    )

    landmarks = face[4:14].reshape(5, 2)

    # YuNet landmark order:
    # right eye, left eye, nose,
    # right mouth, left mouth

    src = landmarks.astype(np.float32)

    dst = np.array([
        [38.2946, 51.6963],
        [73.5318, 51.5014],
        [56.0252, 71.7366],
        [41.5493, 92.3655],
        [70.7299, 92.2041]
    ], dtype=np.float32)

    transform = cv2.estimateAffinePartial2D(
        src,
        dst
    )[0]

    if transform is None:
        return None

    aligned = cv2.warpAffine(
        image,
        transform,
        (112, 112)
    )

    return aligned

In [41]:
# Arceface embedding
# Use the ArcFace API that worked in your previous notebook:
def get_arcface_embedding(face):

    embedding = recognition_model.get(face)

    embedding = np.asarray(
        embedding,
        dtype=np.float32
    )

    embedding /= (
        np.linalg.norm(embedding) + 1e-12
    )

    return embedding

### Complete image recognition

In [42]:
def recognize_image(image):

    # 1. Detect + align
    aligned_face = detect_and_align(image)

    if aligned_face is None:
        return {
            "identity": "No face",
            "similarity": None,
            "status": "NO_FACE"
        }

    # 2. ArcFace embedding
    query_embedding = get_arcface_embedding(
        aligned_face
    )

    # 3. Match against database
    result = recognize_embedding(
        query_embedding,
        recognition_embeddings,
        recognition_metadata,
        FINAL_THRESHOLD
    )

    return result

In [47]:
# Test with one image
test_image_path = ALIGNED_DIR / "Nikhil_Kanbarkar" / "NK25.jpeg"

image = cv2.imread(
    str(test_image_path)
)

result = recognize_image(image)

print("Identity   :", result["identity"])
print("Similarity :", result["similarity"])
print("Status     :", result["status"])

TypeError: ArcFaceONNX.get() missing 1 required positional argument: 'face'

In [48]:
test_image_path = ALIGNED_DIR / "Nikhil_Kanbarkar" / "NK1.jpeg"

image = cv2.imread(
    str(test_image_path)
)

result = recognize_image(image)

print("Identity   :", result["identity"])
print("Similarity :", result["similarity"])
print("Status     :", result["status"])

AttributeError: 'NoneType' object has no attribute 'shape'

# Complete YuNet + ArcFace Recognition Pipeline

In [64]:
# ============================================================
# YUNET + ARCFACE COMPLETE FACE RECOGNITION PIPELINE
# ============================================================

from pathlib import Path

import cv2
import numpy as np
import pandas as pd


# ============================================================
# 1. PATHS
# ============================================================

MODEL_PATH = Path(
    "
    ./models/face_detection_yunet_2026may.onnx"
)

EMBEDDINGS_DIR = Path("./Data/Embeddings")

EMBEDDING_FILE = (
    EMBEDDINGS_DIR / "embeddings_clean.npy"
)

METADATA_FILE = (
    EMBEDDINGS_DIR / "metadata_clean.csv"
)


# ============================================================
# 2. CONFIGURATION
# ============================================================

FINAL_THRESHOLD = 0.36

YUNET_INPUT_SIZE = (320, 320)

YUNET_SCORE_THRESHOLD = 0.6

YUNET_NMS_THRESHOLD = 0.3

YUNET_TOP_K = 5000


# ============================================================
# 3. LOAD DATABASE
# ============================================================

database_embeddings = np.load(
    EMBEDDING_FILE
).astype(np.float32)

database_metadata = pd.read_csv(
    METADATA_FILE
)

print("Database loaded")
print("Embeddings :", database_embeddings.shape)
print("Metadata   :", database_metadata.shape)


# ============================================================
# 4. LOAD YUNET
# ============================================================

detector = cv2.FaceDetectorYN.create(
    model=str(MODEL_PATH),
    config="",
    input_size=YUNET_INPUT_SIZE,
    score_threshold=YUNET_SCORE_THRESHOLD,
    nms_threshold=YUNET_NMS_THRESHOLD,
    top_k=YUNET_TOP_K
)

print("YuNet loaded successfully.")


# ============================================================
# 5. LOAD ARCFACE
# ============================================================

# IMPORTANT:
# Replace this with the SAME ArcFace loading code/path
# that already worked in your previous notebook.

from insightface.model_zoo import get_model

ARCFACE_MODEL_PATH = Path(
    "./models/w600k_r50.onnx"
)

recognition_model = get_model(
    str(ARCFACE_MODEL_PATH)
)

recognition_model.prepare(ctx_id=0)

print("ArcFace loaded successfully.")


# ============================================================
# 6. DETECT + ALIGN FACE
# ============================================================

def detect_and_align(image):

    h, w = image.shape[:2]

    # YuNet must know the actual image size
    detector.setInputSize((w, h))

    _, faces = detector.detect(image)

    if faces is None or len(faces) == 0:
        return None

    # --------------------------------------------------------
    # Select highest-confidence face
    # --------------------------------------------------------

    best_face = max(
        faces,
        key=lambda face: face[14]
    )

    # YuNet:
    # [x, y, w, h,
    #  right_eye,
    #  left_eye,
    #  nose,
    #  right_mouth,
    #  left_mouth,
    #  score]

    landmarks = best_face[
        4:14
    ].reshape(5, 2).astype(np.float32)


    # --------------------------------------------------------
    # ArcFace 112x112 reference landmarks
    # --------------------------------------------------------

    reference_landmarks = np.array([
        [38.2946, 51.6963],
        [73.5318, 51.5014],
        [56.0252, 71.7366],
        [41.5493, 92.3655],
        [70.7299, 92.2041]
    ], dtype=np.float32)


    # --------------------------------------------------------
    # Estimate transformation
    # --------------------------------------------------------

    transform, _ = cv2.estimateAffinePartial2D(
        landmarks,
        reference_landmarks,
        method=cv2.LMEDS
    )

    if transform is None:
        return None


    # --------------------------------------------------------
    # Align face
    # --------------------------------------------------------

    aligned_face = cv2.warpAffine(
        image,
        transform,
        (112, 112)
    )

    return aligned_face


# ============================================================
# 7. ARCFACE EMBEDDING
# ============================================================

def get_embedding(aligned_face):

    embedding = recognition_model.get(
        aligned_face
    )

    embedding = np.asarray(
        embedding,
        dtype=np.float32
    ).reshape(-1)

    # L2 normalization
    embedding /= (
        np.linalg.norm(embedding) + 1e-12
    )

    return embedding


# ============================================================
# 8. FACE MATCHING
# ============================================================

def match_embedding(query_embedding):

    # Cosine similarity because embeddings
    # are L2 normalized

    similarities = np.dot(
        database_embeddings,
        query_embedding
    )

    best_index = int(
        np.argmax(similarities)
    )

    best_similarity = float(
        similarities[best_index]
    )

    best_person = database_metadata.iloc[
        best_index
    ]["person"]


    # --------------------------------------------------------
    # Threshold decision
    # --------------------------------------------------------

    if best_similarity >= FINAL_THRESHOLD:

        identity = best_person
        status = "KNOWN"

    else:

        identity = "Unknown"
        status = "UNKNOWN"


    return {
        "identity": identity,
        "similarity": best_similarity,
        "status": status
    }


# ============================================================
# 9. COMPLETE RECOGNITION FUNCTION
# ============================================================

def recognize_image(image):

    # --------------------------------------------------------
    # Step 1: YuNet detection + alignment
    # --------------------------------------------------------

    aligned_face = detect_and_align(
        image
    )

    if aligned_face is None:

        return {
            "identity": "No face detected",
            "similarity": None,
            "status": "NO_FACE",
            "aligned_face": None
        }


    # --------------------------------------------------------
    # Step 2: ArcFace embedding
    # --------------------------------------------------------

    query_embedding = get_embedding(
        aligned_face
    )


    # --------------------------------------------------------
    # Step 3: Database matching
    # --------------------------------------------------------

    result = match_embedding(
        query_embedding
    )


    result["aligned_face"] = aligned_face

    return result


# ============================================================
# 10. TEST WITH IMAGE
# ============================================================

TEST_IMAGE = Path(
    "./Data/Classmates/Nikhil_Kanbarkar/NK1.jpeg"
)

image = cv2.imread(
    str(TEST_IMAGE)
)

if image is None:

    raise FileNotFoundError(
        f"Could not read image: {TEST_IMAGE}"
    )


result = recognize_image(
    image
)


# ============================================================
# 11. DISPLAY RESULT
# ============================================================

print("\n==============================")
print("FACE RECOGNITION RESULT")
print("==============================")

print(
    "Identity   :",
    result["identity"]
)

print(
    "Similarity :",
    result["similarity"]
)

print(
    "Status     :",
    result["status"]
)

SyntaxError: unterminated string literal (detected at line 17) (1039002291.py, line 17)

In [62]:
# ============================================================
# COMPLETE REAL-TIME FACE RECOGNITION PIPELINE
#
# YuNet
#    ↓
# Face Detection
#    ↓
# 5-Point Landmark Detection
#    ↓
# Face Alignment
#    ↓
# ArcFace / Buffalo_L (w600k_r50.onnx)
#    ↓
# 512-D Embedding
#    ↓
# Cosine Similarity
#    ↓
# Threshold
#    ↓
# KNOWN / UNKNOWN
# ============================================================


from pathlib import Path

import cv2
import numpy as np
import pandas as pd

from insightface.model_zoo import get_model


# ============================================================
# 1. CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

MODEL_PATH = Path(
    "./models/face_detection_yunet_2026may.onnx"
)

ARCFACE_MODEL_PATH = Path(
    "./models/w600k_r50.onnx"
)

EMBEDDING_FILE = Path(
    "./Data/Embeddings/embeddings_clean.npy"
)

METADATA_FILE = Path(
    "./Data/Embeddings/metadata_clean.csv"
)


# ------------------------------------------------------------
# Recognition threshold
# ------------------------------------------------------------

FINAL_THRESHOLD = 0.36


# ------------------------------------------------------------
# YuNet parameters
# ------------------------------------------------------------

YUNET_INPUT_SIZE = (320, 320)

YUNET_SCORE_THRESHOLD = 0.60

YUNET_NMS_THRESHOLD = 0.30

YUNET_TOP_K = 5000


# ------------------------------------------------------------
# Camera
# ------------------------------------------------------------

CAMERA_INDEX = 0


# ============================================================
# 2. CHECK FILES
# ============================================================

required_files = [
    MODEL_PATH,
    ARCFACE_MODEL_PATH,
    EMBEDDING_FILE,
    METADATA_FILE
]

print("\nChecking required files...\n")

for file in required_files:

    print(
        f"{file} -> "
        f"{'FOUND' if file.exists() else 'NOT FOUND'}"
    )

    if not file.exists():

        raise FileNotFoundError(
            f"\nRequired file not found:\n"
            f"{file.resolve()}"
        )


# ============================================================
# 3. LOAD FACE DATABASE
# ============================================================

print("\nLoading face database...")

database_embeddings = np.load(
    EMBEDDING_FILE
).astype(np.float32)

database_metadata = pd.read_csv(
    METADATA_FILE
)


# ------------------------------------------------------------
# Verify database
# ------------------------------------------------------------

if len(database_embeddings) != len(database_metadata):

    raise ValueError(
        "Number of embeddings does not match "
        "number of metadata rows."
    )


print(
    "Embeddings shape :",
    database_embeddings.shape
)

print(
    "Metadata shape   :",
    database_metadata.shape
)

print(
    "Registered images:",
    len(database_embeddings)
)


# ============================================================
# 4. NORMALIZE DATABASE EMBEDDINGS
# ============================================================

database_embeddings = (
    database_embeddings /
    (
        np.linalg.norm(
            database_embeddings,
            axis=1,
            keepdims=True
        )
        + 1e-12
    )
)


# ============================================================
# 5. LOAD YUNET
# ============================================================

print("\nLoading YuNet...")

detector = cv2.FaceDetectorYN.create(
    model=str(MODEL_PATH),
    config="",
    input_size=YUNET_INPUT_SIZE,
    score_threshold=YUNET_SCORE_THRESHOLD,
    nms_threshold=YUNET_NMS_THRESHOLD,
    top_k=YUNET_TOP_K
)

print("YuNet loaded successfully.")


# ============================================================
# 6. LOAD BUFFALO_L ARCFACE
# ============================================================

print("\nLoading ArcFace...")

recognition_model = get_model(
    str(ARCFACE_MODEL_PATH)
)

# CPU
recognition_model.prepare(
    ctx_id=-1
)

print(
    "ArcFace loaded successfully."
)

print(
    "Model:",
    type(recognition_model)
)


# ============================================================
# 7. ARCFACE STANDARD LANDMARKS
# ============================================================

ARC_FACE_REFERENCE = np.array(
    [
        [38.2946, 51.6963],
        [73.5318, 51.5014],
        [56.0252, 71.7366],
        [41.5493, 92.3655],
        [70.7299, 92.2041]
    ],
    dtype=np.float32
)


# ============================================================
# 8. FACE RECOGNITION FUNCTION
# ============================================================

def recognize_frame(frame):

    # --------------------------------------------------------
    # YuNet detection
    # --------------------------------------------------------

    height, width = frame.shape[:2]

    detector.setInputSize(
        (width, height)
    )

    _, faces = detector.detect(frame)

    if faces is None:
        return []


    results = []


    # ========================================================
    # PROCESS EACH FACE
    # ========================================================

    for face in faces:

        # ----------------------------------------------------
        # Bounding box
        # ----------------------------------------------------

        x, y, w, h = face[:4]

        x = int(x)
        y = int(y)
        w = int(w)
        h = int(h)


        # ----------------------------------------------------
        # YuNet landmarks
        # ----------------------------------------------------

        landmarks = (
            face[4:14]
            .reshape(5, 2)
            .astype(np.float32)
        )


        # ----------------------------------------------------
        # Alignment
        # ----------------------------------------------------

        transform, _ = cv2.estimateAffinePartial2D(
            landmarks,
            ARC_FACE_REFERENCE,
            method=cv2.LMEDS
        )

        if transform is None:
            continue


        aligned_face = cv2.warpAffine(
            frame,
            transform,
            (112, 112)
        )


        # ====================================================
        # ARCFACE EMBEDDING
        # ====================================================

        # ArcFaceONNX.get() requires:
        # get(img, face)

        embedding = recognition_model.get(
            aligned_face,
            aligned_face
        )


        embedding = np.asarray(
            embedding,
            dtype=np.float32
        ).reshape(-1)


        # ----------------------------------------------------
        # L2 normalization
        # ----------------------------------------------------

        embedding = embedding / (
            np.linalg.norm(embedding)
            + 1e-12
        )


        # ====================================================
        # COSINE SIMILARITY
        # ====================================================

        similarities = np.dot(
            database_embeddings,
            embedding
        )


        best_index = int(
            np.argmax(similarities)
        )

        best_similarity = float(
            similarities[best_index]
        )


        best_person = str(
            database_metadata.iloc[
                best_index
            ]["person"]
        )


        # ====================================================
        # THRESHOLD
        # ====================================================

        if best_similarity >= FINAL_THRESHOLD:

            identity = best_person
            status = "KNOWN"

        else:

            identity = "UNKNOWN"
            status = "UNKNOWN"


        # ====================================================
        # SAVE RESULT
        # ====================================================

        results.append(
            {
                "bbox": (x, y, w, h),
                "identity": identity,
                "similarity": best_similarity,
                "status": status
            }
        )


    return results

# ============================================================
# REAL-TIME CAMERA - JUPYTER VERSION
# ============================================================

import cv2
import time

from IPython.display import display, Image, clear_output


cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Could not open webcam.")


print("Camera started.")
print("Press Ctrl+C in Jupyter to stop.")


try:

    while True:

        ret, frame = cap.read()

        if not ret:
            print("Failed to read camera frame.")
            break


        # ----------------------------------------------------
        # FACE RECOGNITION
        # ----------------------------------------------------

        results = recognize_frame(frame)


        # ----------------------------------------------------
        # DRAW RESULTS
        # ----------------------------------------------------

        for result in results:

            x, y, w, h = result["bbox"]

            identity = result["identity"]

            similarity = result["similarity"]


            # Bounding box
            cv2.rectangle(
                frame,
                (x, y),
                (x + w, y + h),
                (0, 255, 0),
                2
            )


            # Label
            label = (
                f"{identity} | "
                f"{similarity:.3f}"
            )


            cv2.putText(
                frame,
                label,
                (x, max(y - 10, 25)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                (0, 255, 0),
                2,
                cv2.LINE_AA
            )


        # ----------------------------------------------------
        # Convert OpenCV BGR → JPEG
        # ----------------------------------------------------

        frame_rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        success, buffer = cv2.imencode(
            ".jpg",
            frame_rgb
        )

        if not success:
            continue


        # ----------------------------------------------------
        # Display inside Jupyter
        # ----------------------------------------------------

        clear_output(wait=True)

        display(
            Image(
                data=buffer.tobytes()
            )
        )


        time.sleep(0.03)


except KeyboardInterrupt:

    print("\nCamera stopped by user.")


finally:

    cap.release()

    clear_output(wait=True)

    print("Camera released.")
print("\nCamera stopped.")
print("Recognition pipeline finished.")

Camera released.


AttributeError: 'numpy.ndarray' object has no attribute 'kps'

In [1]:
import cv2
import numpy as np
import pyttsx3
import time
import threading
import queue
import pandas as pd

from insightface.app import FaceAnalysis


# ============================================================
# TEXT TO SPEECH
# ============================================================

engine = pyttsx3.init()

engine.setProperty("rate", 150)
engine.setProperty("volume", 1.0)


# ============================================================
# SPEECH QUEUE
# ============================================================

speech_queue = queue.Queue()


def speech_worker():

    while True:

        name = speech_queue.get()

        if name is None:
            speech_queue.task_done()
            break

        try:

            print(f"[VOICE] {name} detected")

            engine.say(f"{name} detected")
            engine.runAndWait()

        except Exception as e:

            print("Speech error:", e)

        finally:

            speech_queue.task_done()


speech_thread = threading.Thread(
    target=speech_worker,
    daemon=True
)

speech_thread.start()


def speak_name(name):

    if name == "UNKNOWN":
        return

    speech_queue.put(name)


# ============================================================
# PATHS
# ============================================================

YUNET_MODEL = "models/face_detection_yunet_2026may.onnx"

EMBEDDINGS_FILE = "./Data/Embeddings/embeddings_clean.npy"

METADATA_FILE = "./Data/Embeddings/metadata_clean.csv"

# ============================================================
# CAMERA
# ============================================================

CAMERA_INDEX = 0


# ============================================================
# ARCFACE RECOGNITION THRESHOLD
# ============================================================

THRESHOLD = 0.35


# ============================================================
# SPEECH SETTINGS
# ============================================================

# Speak immediately when a person appears.
# Then repeat after this many seconds if continuously visible.

REPEAT_TIME = 30.0


# ============================================================
# PERSON PRESENCE SETTINGS
# ============================================================

# Maximum distance for matching the same face
# between consecutive frames.

MAX_TRACK_DISTANCE = 100


# If a face disappears for more than this time,
# consider that person to have left.

TRACK_LOST_TIMEOUT = 1.0


# ============================================================
# LOAD STORED EMBEDDINGS
# ============================================================

print()
print("==============================================")
print("FACE DATABASE")
print("==============================================")


data = np.load(
    EMBEDDINGS_FILE,
    allow_pickle=False
)

database_embeddings = data.astype(np.float32)

print("Embeddings shape:", database_embeddings.shape)



# ============================================================
# LOAD FACE DATABASE
# ============================================================

database_embeddings = np.load(
    EMBEDDINGS_FILE,
    allow_pickle=False
).astype(np.float32)


database_metadata = pd.read_csv(
    METADATA_FILE
)

print("\n==============================================")
print("FACE DATABASE")
print("==============================================")

print("Embeddings shape:", database_embeddings.shape)
print("Metadata shape  :", database_metadata.shape)


# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

if len(database_embeddings) != len(database_metadata):
    raise ValueError(
        f"Mismatch: "
        f"{len(database_embeddings)} embeddings vs "
        f"{len(database_metadata)} metadata rows"
    )


# ------------------------------------------------------------
# Person labels come from metadata CSV
# ------------------------------------------------------------

database_labels = database_metadata["person"].astype(str).to_numpy()

print("Database labels:", len(database_labels))
print("Unique persons :", database_metadata["person"].nunique())

print("\nPersons:")
print(database_metadata["person"].value_counts())


print("Embeddings:", database_embeddings.shape)

print("Labels:", len(database_labels))


# ============================================================
# NORMALIZE DATABASE EMBEDDINGS
# ============================================================

database_norms = np.linalg.norm(
    database_embeddings,
    axis=1,
    keepdims=True
)

database_embeddings = (
    database_embeddings /
    np.maximum(database_norms, 1e-10)
)


# ============================================================
# LOAD ARCFACE
# ============================================================

print()
print("Loading ArcFace...")


app = FaceAnalysis(
    name="buffalo_l",
    providers=["CPUExecutionProvider"]
)


app.prepare(
    ctx_id=0,
    det_size=(640, 640)
)


recognition_model = app.models["recognition"]


print("ArcFace recognition model loaded.")


# ============================================================
# LOAD YUNET
# ============================================================

print()
print("Loading YuNet...")


yunet = cv2.FaceDetectorYN.create(
    YUNET_MODEL,
    "",
    (320, 320),
    0.6,
    0.3,
    5000
)


print("YuNet model loaded.")


# ============================================================
# CAMERA
# ============================================================

print()
print("Opening camera...")


camera = cv2.VideoCapture(
    CAMERA_INDEX,
    cv2.CAP_DSHOW
)


if not camera.isOpened():

    print("ERROR: Camera could not be opened.")

    speech_queue.put(None)

    raise SystemExit(1)


# ============================================================
# CAMERA SETTINGS
# ============================================================

camera.set(
    cv2.CAP_PROP_FRAME_WIDTH,
    640
)

camera.set(
    cv2.CAP_PROP_FRAME_HEIGHT,
    480
)


# ============================================================
# TEST FIRST FRAME
# ============================================================

ret, test_frame = camera.read()


if not ret or test_frame is None:

    print()
    print("ERROR: Camera opened but could not read a frame.")
    print()

    print("Possible causes:")
    print("1. Another application is using the camera.")
    print("2. Windows camera permission is disabled.")
    print("3. Wrong camera index.")
    print("4. Camera driver problem.")
    print()

    camera.release()

    speech_queue.put(None)

    raise SystemExit(1)


print()
print("==============================================")
print("CAMERA STARTED")
print("==============================================")

print(
    "Camera resolution:",
    test_frame.shape[1],
    "x",
    test_frame.shape[0]
)

print()
print("Speech behavior:")
print(" - New person -> speak immediately")
print(" - Same person continuously visible -> repeat every 30 seconds")
print(" - Person leaves and returns -> speak immediately again")
print()
print("Press Q to quit.")
print()


# ============================================================
# TRACKING DATA
# ============================================================

tracks = {}


# ============================================================
# NEXT TRACK ID
# ============================================================

next_track_id = 0


# ============================================================
# CREATE NEW TRACK
# ============================================================

def create_track(
    center,
    name,
    similarity,
    bbox
):

    global next_track_id

    track_id = next_track_id

    next_track_id += 1

    current_time = time.time()

    tracks[track_id] = {

        "center": center,

        "name": name,

        "similarity": similarity,

        # When this person first appeared
        "first_seen": current_time,

        # Last frame in which this person was detected
        "last_seen": current_time,

        # Last time we spoke this person's name
        "last_spoken": 0.0,

        "bbox": bbox

    }

    return track_id


# ============================================================
# FIND CLOSEST EXISTING TRACK
# ============================================================

def find_matching_track(
    center,
    used_tracks
):

    best_track_id = None

    best_distance = MAX_TRACK_DISTANCE

    cx, cy = center

    for track_id, track in tracks.items():

        # Do not use one track for two faces
        if track_id in used_tracks:
            continue

        tx, ty = track["center"]

        distance = np.sqrt(
            (cx - tx) ** 2 +
            (cy - ty) ** 2
        )

        if distance < best_distance:

            best_distance = distance

            best_track_id = track_id

    return best_track_id


# ============================================================
# REAL-TIME LOOP
# ============================================================

while True:

    # ========================================================
    # READ CAMERA FRAME
    # ========================================================

    ret, frame = camera.read()


    if not ret or frame is None:

        print("ERROR: Could not read frame.")

        break


    # ========================================================
    # FRAME SIZE
    # ========================================================

    height, width = frame.shape[:2]


    # ========================================================
    # TELL YUNET CURRENT FRAME SIZE
    # ========================================================

    yunet.setInputSize(
        (width, height)
    )


    # ========================================================
    # FACE DETECTION
    # ========================================================

    _, detections = yunet.detect(frame)


    # ========================================================
    # TRACKS USED IN THIS FRAME
    # ========================================================

    used_tracks = set()


    # ========================================================
    # CURRENT TRACK IDS
    # ========================================================

    current_track_ids = set()


    # ========================================================
    # PROCESS DETECTED FACES
    # ========================================================

    if detections is not None:

        for detection in detections:

            # =================================================
            # BOUNDING BOX
            # =================================================

            x, y, w, h = detection[:4]

            x = int(x)
            y = int(y)
            w = int(w)
            h = int(h)


            # =================================================
            # KEEP BOUNDING BOX INSIDE IMAGE
            # =================================================

            x1 = max(0, x)
            y1 = max(0, y)

            x2 = min(width, x + w)
            y2 = min(height, y + h)


            if x2 <= x1 or y2 <= y1:

                continue


            # =================================================
            # FACE CENTER
            # =================================================

            center_x = int(
                (x1 + x2) / 2
            )

            center_y = int(
                (y1 + y2) / 2
            )

            center = (
                center_x,
                center_y
            )


            # =================================================
            # FACE CROP
            # =================================================

            face_crop = frame[
                y1:y2,
                x1:x2
            ]


            if face_crop.size == 0:

                continue


            # =================================================
            # ARCFACE EMBEDDING
            # =================================================

            try:

                embedding = recognition_model.get_feat(
                    face_crop
                )


                embedding = np.asarray(
                    embedding,
                    dtype=np.float32
                ).flatten()


                # =============================================
                # NORMALIZE EMBEDDING
                # =============================================

                norm = np.linalg.norm(
                    embedding
                )


                if norm == 0:

                    continue


                embedding = (
                    embedding /
                    norm
                )


            except Exception as e:

                print(
                    "Embedding error:",
                    e
                )

                continue


            # =================================================
            # COSINE SIMILARITY
            # =================================================

            similarities = np.dot(
                database_embeddings,
                embedding
            )


            best_index = np.argmax(
                similarities
            )


            best_similarity = float(
                similarities[best_index]
            )


            predicted_name = database_labels[
                best_index
            ]


            # =================================================
            # RECOGNITION DECISION
            # =================================================

            if best_similarity >= THRESHOLD:

                name = str(
                    predicted_name
                )

            else:

                name = "UNKNOWN"


            # =================================================
            # FIND EXISTING TRACK
            # =================================================

            track_id = find_matching_track(
                center,
                used_tracks
            )


            # =================================================
            # NEW TRACK / NEW PERSON
            # =================================================

            if track_id is None:

                track_id = create_track(
                    center,
                    name,
                    best_similarity,
                    (
                        x1,
                        y1,
                        x2,
                        y2
                    )
                )


                used_tracks.add(
                    track_id
                )


                current_track_ids.add(
                    track_id
                )


                track = tracks[
                    track_id
                ]


                # =============================================
                # NEW PERSON
                # =============================================

                if name != "UNKNOWN":

                    print()
                    print(
                        "================================"
                    )

                    print(
                        f"NEW PERSON DETECTED: {name}"
                    )

                    print(
                        f"Similarity: {best_similarity:.4f}"
                    )

                    print(
                        "Speaking name..."
                    )

                    print(
                        "================================"
                    )


                    # =========================================
                    # SPEAK IMMEDIATELY
                    # =========================================

                    track["last_spoken"] = time.time()

                    speak_name(name)


            # =================================================
            # EXISTING TRACK
            # =================================================

            else:

                used_tracks.add(
                    track_id
                )


                current_track_ids.add(
                    track_id
                )


                track = tracks[
                    track_id
                ]


                # =============================================
                # UPDATE POSITION
                # =============================================

                track["center"] = center

                track["bbox"] = (
                    x1,
                    y1,
                    x2,
                    y2
                )


                track["last_seen"] = time.time()


                # =============================================
                # UPDATE RECOGNITION
                # =============================================

                if name != "UNKNOWN":

                    track["name"] = name

                    track["similarity"] = (
                        best_similarity
                    )


                # =============================================
                # 30 SECOND REPEAT
                # =============================================

                current_time = time.time()


                if (
                    track["name"] != "UNKNOWN"
                    and
                    current_time -
                    track["last_spoken"]
                    >= REPEAT_TIME
                ):

                    print()
                    print(
                        "--------------------------------"
                    )

                    print(
                        f"30 SECOND REPEAT: "
                        f"{track['name']}"
                    )

                    print(
                        "Speaking name..."
                    )

                    print(
                        "--------------------------------"
                    )


                    # Update BEFORE putting into queue
                    # so another frame cannot queue
                    # the same person repeatedly.

                    track["last_spoken"] = current_time


                    speak_name(
                        track["name"]
                    )


            # =================================================
            # DISPLAY INFORMATION
            # =================================================

            track = tracks[
                track_id
            ]


            display_name = track[
                "name"
            ]


            display_similarity = track[
                "similarity"
            ]


            # =================================================
            # LABEL
            # =================================================

            if display_name == "UNKNOWN":

                label = (
                    f"UNKNOWN "
                    f"({display_similarity:.2f})"
                )

            else:

                label = (
                    f"{display_name} "
                    f"({display_similarity:.2f})"
                )


            # =================================================
            # DRAW RECTANGLE
            # =================================================

            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )


            # =================================================
            # DRAW NAME
            # =================================================

            cv2.putText(
                frame,
                label,
                (
                    x1,
                    max(
                        y1 - 10,
                        30
                    )
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2
            )


            # =================================================
            # DRAW TRACKING ID
            # =================================================

            cv2.putText(
                frame,
                f"ID: {track_id}",
                (
                    x1,
                    min(
                        y2 + 25,
                        height - 10
                    )
                ),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 0),
                2
            )


    # ========================================================
    # REMOVE LOST TRACKS
    # ========================================================

    current_time = time.time()

    tracks_to_remove = []


    for track_id, track in tracks.items():

        time_since_seen = (
            current_time -
            track["last_seen"]
        )


        if time_since_seen > TRACK_LOST_TIMEOUT:

            tracks_to_remove.append(
                track_id
            )


    # ========================================================
    # DELETE LOST TRACKS
    # ========================================================

    for track_id in tracks_to_remove:

        old_name = tracks[
            track_id
        ]["name"]


        print(
            f"Person left frame: "
            f"{old_name}"
        )


        del tracks[
            track_id
        ]


    # ========================================================
    # DISPLAY CAMERA
    # ========================================================

    cv2.imshow(
        "YuNet + ArcFace Real-Time Face Recognition",
        frame
    )


    # ========================================================
    # QUIT
    # ========================================================

    key = cv2.waitKey(1) & 0xFF


    if key == ord("q"):

        break


# ============================================================
# CLEANUP
# ============================================================

camera.release()

cv2.destroyAllWindows()


# ============================================================
# STOP SPEECH WORKER
# ============================================================

speech_queue.put(None)

speech_thread.join(
    timeout=2
)


print()
print("Camera stopped.")
print("Program finished.")


FACE DATABASE
Embeddings shape: (211, 512)

FACE DATABASE
Embeddings shape: (211, 512)
Metadata shape  : (211, 3)
Database labels: 211
Unique persons : 14

Persons:
person
Sahil_Patil            29
Nikhil_Kanbarkar       25
Rohit_Patil            24
Dhurvankur_            21
Abhishek_Halagi        17
Harshad_Bhairatkar     16
Darshan_Khadakhadi     15
Suraj_Mallur           14
Darshan_Netegal        12
Vishwanath_Muchandi    11
Mahesh_Oulkar          10
Pranav_Kavale           6
Rohan_Kamble            6
Sahil_Kurbet            5
Name: count, dtype: int64
Embeddings: (211, 512)
Labels: 211

Loading ArcFace...
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\Nikhil/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\Nikhil/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['N

error: OpenCV(5.0.0) D:\a\opencv-python\opencv-python\opencv\modules\highgui\src\window.cpp:1215: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'showImageImpl'
